### Notebook 6 — Train, tune, evaluate


## 1. Load Processed Data

The processed training, validation, and test datasets generated in Notebook 5 are loaded here.

The test set is loaded for later use, but it will not be used during model selection or tuning.

In [10]:
# Notebook 6 — Model Training, Tuning & Evaluation

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
)

os.makedirs("../artifacts", exist_ok=True)

# Load processed data
X_train = np.load("../artifacts/X_train_processed.npy")
X_val = np.load("../artifacts/X_val_processed.npy")
X_test = np.load("../artifacts/X_test_processed.npy")

y_train = np.load("../artifacts/y_train.npy")
y_val = np.load("../artifacts/y_val.npy")
y_test = np.load("../artifacts/y_test.npy")

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (67533, 54)
Validation: (14471, 54)
Test: (14472, 54)


## 2. Simple Baseline

A DummyClassifier is used as a simple baseline.

The model always predicts the most frequent class. This provides a reference point that the machine learning models should outperform.

Because the target is imbalanced, accuracy alone is not sufficient. Therefore, precision, recall, and F1-score are also reported.

In [11]:
# Baseline model
baseline = DummyClassifier(strategy="most_frequent")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_val)

print("Baseline Validation Results")
print("----------------------------")
print("Accuracy:", accuracy_score(y_val, baseline_pred))
print("Precision:", precision_score(y_val, baseline_pred, zero_division=0))
print("Recall:", recall_score(y_val, baseline_pred, zero_division=0))
print("F1:", f1_score(y_val, baseline_pred, zero_division=0))

Baseline Validation Results
----------------------------
Accuracy: 0.9465828208140419
Precision: 0.0
Recall: 0.0
F1: 0.0


### Baseline Finding

The baseline achieves high accuracy because most orders are on time. However, it has zero recall and zero F1-score because it does not identify any late orders.

This confirms that accuracy alone is misleading for this imbalanced classification problem.

## 3. Target Distribution

The target distribution is inspected to understand the degree of class imbalance in the training and validation sets.

The split is time-based, so the class distribution may change over time. This is expected because the late-order rate was observed to increase across the dataset period.

In [12]:
print("Training target distribution:")
print(pd.Series(y_train).value_counts())
print("\nTraining target proportions:")
print(pd.Series(y_train).value_counts(normalize=True))

print("\nValidation target distribution:")
print(pd.Series(y_val).value_counts())
print("\nValidation target proportions:")
print(pd.Series(y_val).value_counts(normalize=True))

Training target distribution:
0    61436
1     6097
Name: count, dtype: int64

Training target proportions:
0    0.909718
1    0.090282
Name: proportion, dtype: float64

Validation target distribution:
0    13698
1      773
Name: count, dtype: int64

Validation target proportions:
0    0.946583
1    0.053417
Name: proportion, dtype: float64


### Finding

The target is imbalanced, with late orders representing a minority of the observations.

The validation set also has a lower late-order rate than the training set. This difference is expected from the chronological split used in the previous notebooks and reflects the temporal change in late-order behavior.

## 4. Random Forest Baseline

A Random Forest classifier is trained as the first machine learning baseline.

Class weighting is set to `balanced` to give more importance to the minority class (late orders).

In [13]:
# Random Forest baseline

rf_baseline = RandomForestClassifier(
    n_estimators=200, random_state=42, n_jobs=-1, class_weight="balanced"
)

rf_baseline.fit(X_train, y_train)

rf_val_pred = rf_baseline.predict(X_val)

print("Random Forest Validation Results")
print("----------------------------------")
print("Accuracy:", accuracy_score(y_val, rf_val_pred))
print("Precision:", precision_score(y_val, rf_val_pred, zero_division=0))
print("Recall:", recall_score(y_val, rf_val_pred, zero_division=0))
print("F1:", f1_score(y_val, rf_val_pred, zero_division=0))

Random Forest Validation Results
----------------------------------
Accuracy: 0.9464446133646603
Precision: 0.0
Recall: 0.0
F1: 0.0


### Random Forest Baseline Finding

At the default classification threshold of 0.5, the Random Forest predicts almost all validation orders as on time.

Although the model is trained with class balancing, the default threshold does not provide useful recall for the minority class. Therefore, the predicted probabilities will be inspected before deciding whether threshold tuning is needed.

## 5. Random Forest Prediction Distribution

The predicted classes are inspected to determine how the baseline Random Forest behaves on the validation set.

In [14]:
print("Random Forest Validation Predictions:")
print(pd.Series(rf_val_pred).value_counts())

print("\nPrediction proportions:")
print(pd.Series(rf_val_pred).value_counts(normalize=True))

Random Forest Validation Predictions:
0    14469
1        2
Name: count, dtype: int64

Prediction proportions:
0    0.999862
1    0.000138
Name: proportion, dtype: float64


### Finding

The Random Forest predicts almost all validation observations as class 0 at the default threshold of 0.5.

This suggests that the default threshold is too conservative for detecting late orders, so the predicted probabilities are examined next.

## 6. Predicted Probability Analysis

Instead of using only the final predicted class, the predicted probability of a late order is examined.

This helps determine whether the model is assigning meaningful probability scores to late orders even when the default 0.5 threshold does not classify them as late.

In [15]:
# Inspect predicted probabilities

rf_val_proba = rf_baseline.predict_proba(X_val)[:, 1]

print("Predicted probability summary:")
print(pd.Series(rf_val_proba).describe())

print("\nHighest predicted probabilities:")
print(np.sort(rf_val_proba)[-20:])

Predicted probability summary:
count    14471.000000
mean         0.078283
std          0.066550
min          0.000000
25%          0.030000
50%          0.060000
75%          0.105000
max          0.510000
dtype: float64

Highest predicted probabilities:
[0.40954859 0.41       0.41       0.415      0.42       0.42
 0.42       0.425      0.425      0.425      0.43       0.43
 0.43       0.435      0.445      0.455      0.47       0.49
 0.51       0.51      ]


### Finding

The predicted probabilities show that the model assigns non-zero probabilities to many observations, but most probabilities are below 0.5.

Therefore, the lack of positive predictions at the default threshold is mainly related to the classification threshold rather than the complete absence of predictive signal.

## 7. Classification Threshold Tuning

The classification threshold is tuned using the validation set.

Several thresholds are evaluated using precision, recall, and F1-score.

F1-score is used as the main selection metric because the target is imbalanced and both precision and recall are important.

In [16]:
# Evaluate different classification thresholds on the validation set

thresholds = np.arange(0.05, 0.51, 0.05)

threshold_results = []

for threshold in thresholds:
    val_pred_threshold = (rf_val_proba >= threshold).astype(int)

    threshold_results.append(
        {
            "Threshold": threshold,
            "Precision": precision_score(y_val, val_pred_threshold, zero_division=0),
            "Recall": recall_score(y_val, val_pred_threshold, zero_division=0),
            "F1": f1_score(y_val, val_pred_threshold, zero_division=0),
        }
    )

threshold_results = pd.DataFrame(threshold_results)

threshold_results

,Threshold,Precision,Recall,F1
0,0.05,0.065440,0.727038,0.120073
1,0.10,0.087376,0.445019,0.146072
2,0.15,0.108401,0.258732,0.152788
3,0.20,0.119632,0.151358,0.133638
4,0.25,0.134703,0.076326,0.097440
5,0.30,0.160221,0.037516,0.060797
6,0.35,0.173913,0.015524,0.028504
7,0.40,0.214286,0.007762,0.014981
8,0.45,0.000000,0.000000,0.000000
9,0.50,0.000000,0.000000,0.000000


### Finding

The best validation F1-score is obtained at a threshold of 0.15:

- Precision: 0.1084
- Recall: 0.2587
- F1-score: 0.1528

This is substantially better than the default threshold of 0.5, which produced an F1-score of 0.

The threshold of 0.15 will be considered during final model selection.

## 8. Random Forest Hyperparameter Tuning

The Random Forest hyperparameters are tuned using cross-validation on the training set.

F1-score is used as the optimization metric because the target variable is imbalanced.

The validation set remains separate and will be used later to compare the tuned model and select the classification threshold.

In [17]:
from sklearn.model_selection import GridSearchCV

# Hyperparameter tuning using F1-score
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

rf_model = RandomForestClassifier(random_state=42, n_jobs=-1, class_weight="balanced")

grid_search = GridSearchCV(
    estimator=rf_model, param_grid=param_grid, scoring="f1", cv=3, n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)

print("Best parameters:")
print(grid_search.best_params_)

print("\nBest cross-validation F1:")
print(grid_search.best_score_)

Fitting 3 folds for each of 24 candidates, totalling 72 fits
Best parameters:
{'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 100}

Best cross-validation F1:
0.09397904208672643


### Hyperparameter Tuning Finding

The best Random Forest configuration was selected based on the highest cross-validation F1-score obtained on the training data.

The selected configuration is:

- `n_estimators`: 100
- `max_depth`: 10
- `min_samples_split`: 5
- `min_samples_leaf`: 1

The best cross-validation F1-score was approximately 0.094.

The validation set was not used during this hyperparameter search. It will now be used to evaluate the tuned model on unseen data and compare its performance with the baseline model.

In [18]:
# Evaluate the tuned Random Forest on the validation set

best_rf = grid_search.best_estimator_

val_tuned_pred = best_rf.predict(X_val)

print("Tuned Random Forest Validation Results")
print("---------------------------------------")
print("Accuracy:", accuracy_score(y_val, val_tuned_pred))
print("Precision:", precision_score(y_val, val_tuned_pred, zero_division=0))
print("Recall:", recall_score(y_val, val_tuned_pred, zero_division=0))
print("F1:", f1_score(y_val, val_tuned_pred, zero_division=0))

Tuned Random Forest Validation Results
---------------------------------------
Accuracy: 0.6926266325754958
Precision: 0.08530805687203792
Recall: 0.4890038809831824
F1: 0.1452728670253651


### Validation Finding

The tuned Random Forest improves the detection of late orders compared with the baseline model.

At the default classification threshold of 0.5, the tuned model achieves a recall of approximately 0.489, meaning that it identifies about 49% of the late orders in the validation set.

Although accuracy decreases compared with the simple baseline, this is expected for an imbalanced classification problem because the model is making more positive predictions instead of predicting almost all orders as on time.

The F1-score is used as the main metric for further model selection. Therefore, the classification threshold will be tuned on the validation set to find a better balance between precision and recall.

In [19]:
# Tune the classification threshold for the tuned Random Forest

tuned_val_proba = best_rf.predict_proba(X_val)[:, 1]

thresholds = np.arange(0.05, 0.51, 0.05)

tuned_threshold_results = []

for threshold in thresholds:
    val_pred_threshold = (tuned_val_proba >= threshold).astype(int)

    tuned_threshold_results.append(
        {
            "Threshold": threshold,
            "Precision": precision_score(y_val, val_pred_threshold, zero_division=0),
            "Recall": recall_score(y_val, val_pred_threshold, zero_division=0),
            "F1": f1_score(y_val, val_pred_threshold, zero_division=0),
        }
    )

tuned_threshold_results = pd.DataFrame(tuned_threshold_results)

tuned_threshold_results

,Threshold,Precision,Recall,F1
0,0.05,0.053417,1.000000,0.101417
1,0.10,0.053445,0.997413,0.101454
2,0.15,0.054807,0.979301,0.103805
3,0.20,0.056599,0.935317,0.106739
4,0.25,0.057703,0.842173,0.108005
5,0.30,0.060187,0.759379,0.111533
6,0.35,0.065105,0.668823,0.118660
7,0.40,0.069377,0.624838,0.124887
8,0.45,0.074314,0.592497,0.132065
9,0.50,0.085308,0.489004,0.145273


### Threshold Tuning Finding

For the tuned Random Forest, the highest F1-score among the evaluated thresholds was 0.1453 at a threshold of 0.50.

At this threshold, the model achieved:
- Precision: 0.0853
- Recall: 0.4890
- F1-score: 0.1453

The tuned model provides substantially higher recall than the original Random Forest baseline. However, its best validation F1-score is slightly lower than the original Random Forest's best validation F1-score of 0.1528 obtained at a threshold of 0.15.

Therefore, both models will be compared using their best validation F1-score before selecting the final model.

## 9. Final Model Selection

The models were compared using the validation F1-score.

The original Random Forest with a classification threshold of 0.15 achieved the highest validation F1-score of 0.1528, compared with 0.1453 for the tuned Random Forest.

Therefore, the original Random Forest with a threshold of 0.15 is selected as the final model.

In [20]:
# Select the final model and classification threshold

final_model = rf_baseline
final_threshold = 0.15

print("Final model: Random Forest")
print("Final threshold:", final_threshold)
print("Validation F1:", 0.152788)

Final model: Random Forest
Final threshold: 0.15
Validation F1: 0.152788


## 10. Final Evaluation on the Test Set

The final model and classification threshold were selected using the validation set.

The test set is now used once to provide an unbiased estimate of the final model's performance on unseen data.

No further model or threshold tuning will be performed using the test results.

In [21]:
# Final evaluation on the test set

test_proba = final_model.predict_proba(X_test)[:, 1]

test_pred = (test_proba >= final_threshold).astype(int)

final_results = {
    "Accuracy": accuracy_score(y_test, test_pred),
    "Precision": precision_score(y_test, test_pred, zero_division=0),
    "Recall": recall_score(y_test, test_pred, zero_division=0),
    "F1": f1_score(y_test, test_pred, zero_division=0),
}

print("Final Test Results")
print("------------------")

for metric, value in final_results.items():
    print(f"{metric}: {value:.4f}")

print("\nClassification Report")
print("---------------------")
print(classification_report(y_test, test_pred, zero_division=0))

Final Test Results
------------------
Accuracy: 0.8127
Precision: 0.0489
Recall: 0.0993
F1: 0.0655

Classification Report
---------------------
              precision    recall  f1-score   support

           0       0.93      0.86      0.90     13515
           1       0.05      0.10      0.07       957

    accuracy                           0.81     14472
   macro avg       0.49      0.48      0.48     14472
weighted avg       0.87      0.81      0.84     14472



### Final Test Evaluation Finding

The final Random Forest model achieved an F1-score of 0.0655 on the unseen test set, with a precision of 0.0489 and a recall of 0.0993 for late orders.

The model detects approximately 10% of the late orders, but the low precision indicates that many of its late-order predictions are false positives.

The test performance is lower than the validation performance, indicating that the model generalizes poorly to the unseen test period. This may be related to the temporal nature of the dataset and the change in late-order rates over time.

Although the final model does not achieve strong predictive performance, the evaluation provides an unbiased estimate of its performance on unseen data. The test set was used only once after completing model selection and tuning.

## 11. Save Final Model and Results

The final trained model and the final test-set results are saved as artifacts.

These artifacts can be reused later for deployment or further analysis without retraining the model.

In [23]:
# Save the final trained model

joblib.dump(final_model, "../artifacts/final_model.joblib")

# Add model information to the results summary
results_summary = {
    "model": "Random Forest",
    "threshold": final_threshold,
    "accuracy": final_results["Accuracy"],
    "precision": final_results["Precision"],
    "recall": final_results["Recall"],
    "f1": final_results["F1"],
}

results_df = pd.DataFrame([results_summary])

results_df.to_csv("../artifacts/final_results.csv", index=False)

print("Final model saved to:")
print("../artifacts/final_model.joblib")

print("\nResults summary saved to:")
print("../artifacts/final_results.csv")

print("\nFinal results:")
print(results_df)

Final model saved to:
../artifacts/final_model.joblib

Results summary saved to:
../artifacts/final_results.csv

Final results:
           model  threshold  accuracy  precision    recall        f1
0  Random Forest       0.15  0.812742   0.048893  0.099269  0.065517


## 12. Final Summary & Findings

### Summary

- A DummyClassifier baseline achieved high accuracy but an F1-score of 0, confirming that accuracy is misleading for the imbalanced target.
- A Random Forest baseline was trained using balanced class weights.
- Classification threshold tuning improved the baseline validation F1-score from 0 at the default threshold of 0.50 to 0.1528 at a threshold of 0.15.
- Random Forest hyperparameters were tuned using 3-fold cross-validation with F1-score as the optimization metric.
- The tuned Random Forest achieved a best validation F1-score of 0.1453 among the evaluated thresholds.
- The original Random Forest with a threshold of 0.15 was selected as the final model because it achieved the highest validation F1-score.
- On the unseen test set, the final model achieved an F1-score of 0.0655, with a precision of 0.0489 and recall of 0.0993.
- The test set was evaluated only once after model selection and tuning.
- The final trained model and results summary were saved as artifacts.

### Conclusion
The final model provides limited performance in detecting late orders, particularly on the unseen test period. The lower test performance compared with validation indicates that the model does not generalize well to the future test period.

These results highlight the difficulty of predicting the minority class under temporal distribution changes and demonstrate why F1-score, precision, and recall are more informative than accuracy alone for this imbalanced classification problem.